In [46]:
import pandas as pd
import os
from glob import glob
from datetime import datetime
from pykrx import stock
from dateutil.relativedelta import *
import OpenDartReader as odr
import FinanceDataReader as fdr

def load_fsfile(filelist, yr):
    """
    특정 연도(year)에 해당하는 파일만 읽어와서 con_df, sol_df 를 만든 뒤
    종목코드 리스트를 뽑아 반환
    """
    con_df = pd.DataFrame()
    sol_df = pd.DataFrame()

    # 연도에 해당하는 파일만 처리
    for each_f in filelist:
        if str(yr) not in each_f:
            continue

        print(f"📂 파일 읽는중: {each_f}")
        try:
            temp_df = pd.read_csv(each_f, sep="\t", encoding='cp949')
            temp_df.columns = temp_df.columns.str.strip()  # 공백제거
            # 연결재무제표와 별도재무제표 분류 예시 (필요에 따라 수정)
            if '연결' in each_f:
                con_df = pd.concat([con_df, temp_df])
            else:
                sol_df = pd.concat([sol_df, temp_df])
        except Exception as e:
            print(f"🚨 파일 읽기 오류: {each_f}, 에러: {e}")

    # ✅ KeyError 방지 및 디버깅 출력
    if not con_df.empty:
        print(f"🔍 con_df.shape={con_df.shape}, columns={list(con_df.columns)}")
    else:
        print("⚠️ con_df 비어있음")

    if not sol_df.empty:
        print(f"🔍 sol_df.shape={sol_df.shape}, columns={list(sol_df.columns)}")
    else:
        print("⚠️ sol_df 비어있음")

    # 종목코드 추출
    if '종목코드' in con_df.columns:
        con_corp = list(set(con_df['종목코드']))
    else:
        print(f"⚠️ con_df에 '종목코드' 없음: {list(con_df.columns)}")
        con_corp = []

    if '종목코드' in sol_df.columns:
        sol_corp = list(set(sol_df['종목코드']))
    else:
        print(f"⚠️ sol_df에 '종목코드' 없음: {list(sol_df.columns)}")
        sol_corp = []

    only_sol_corp = [j for j in sol_corp if j not in con_corp]

    # 나중에 최종 데이터프레임 만들때 활용할 수 있도록 반환
    return pd.concat([con_df, sol_df]), con_corp, sol_corp, only_sol_corp


def make_finance_df(folder_path):
    """
    전체 폴더에서 연도별 재무데이터를 로드하여 하나의 통합 DataFrame을 생성
    """
    # 모든 파일 리스트업
    filelist = [os.path.join(folder_path, fname) for fname in os.listdir(folder_path) if fname.endswith('.txt')]
    print(f"📂 전체 파일수: {len(filelist)}")

    yr_list = [2023, 2024]  # 필요시 더 추가

    all_data = pd.DataFrame()

    for yr in yr_list:
        print(f"\n🚀 {yr} 데이터 처리중...")
        fs_df, con_corp, sol_corp, only_sol_corp = load_fsfile(filelist, yr)
        print(f"✅ {yr} fs_df shape={fs_df.shape}")

        # 통합 DataFrame에 누적
        all_data = pd.concat([all_data, fs_df], ignore_index=True)

    print(f"\n🎉 최종 통합 데이터 shape={all_data.shape}")
    return all_data

def match_dart_code(fs_df):
    fs_df['key_var'] = ''
    fs_df.loc[fs_df['항목명'] == 'ifrs_ProfitLoss', 'key_var'] = '당기순이익'
    fs_df.loc[fs_df['항목명'] == 'ifrs-full_ProfitLoss', 'key_var'] = '당기순이익'
    fs_df.loc[fs_df['항목명'] == 'ifrs_Revenue', 'key_var'] = '매출액'
    fs_df.loc[fs_df['항목명'] == 'ifrs-full_Revenue', 'key_var'] = '매출액'
    fs_df.loc[fs_df['항목명'] == 'ifrs_Equity', 'key_var'] = '자본총계'
    fs_df.loc[fs_df['항목명'] == 'ifrs-full_Equity', 'key_var'] = '자본총계'
    fs_df.loc[fs_df['항목명'] == 'ifrs_CashFlowsFromUsedInOperatingActivities', 'key_var'] = '영업활동현금흐름'
    fs_df.loc[fs_df['항목명'] == 'ifrs-full_CashFlowsFromUsedInOperatingActivities', 'key_var'] = '영업활동현금흐름'
    return fs_df

def adjust_file(fs_df):
    fs_df = fs_df.loc[fs_df['key_var'] != ""]
    fs_df = fs_df[fs_df['종목코드'].isna() == False]
    fs_df = fs_df.drop_duplicates(subset=['종목코드', 'key_var'], keep='last')
    fs_df.reset_index(inplace=True, drop=True)
    return fs_df

def pivot_df(fs_df, yr):
    fs_df['key_var'] = fs_df['key_var'] + '_' + str(yr)
    fs_df['당기'] = fs_df['당기'].apply(lambda x: int(x.replace(',', '')))
    fs_pivotdf = fs_df.pivot_table(index=['종목코드'], columns='key_var', values='당기')
    fs_pivotdf.reset_index(inplace=True)
    fs_pivotdf.sort_values(by='종목코드', inplace=True)
    return fs_pivotdf

def change_colname(fs_df, base_yr):
    collist_dict = {}
    collist = list(fs_df.columns)
    collist_dict[f'{base_yr}'] = [j for j in collist if str(base_yr) in j]
    for pyr in [1, 2, 3]:
        collist_dict[f'{base_yr - pyr}'] = [j for j in collist if str(base_yr - pyr) in j]

    adj_cols = ['종목코드', '종목명']
    for each_col in collist_dict.values():
        adj_cols = adj_cols + each_col

    fs_df_adj = fs_df[adj_cols]

    for curc in collist_dict[f'{base_yr}']:
        fs_df_adj.rename(columns={curc: curc.replace(f'_{base_yr}', '')}, inplace=True)

    for pyr in [1, 2, 3]:
        for prec in collist_dict[f'{base_yr - pyr}']:
            fs_df_adj.rename(columns={prec: f'{pyr}Y{prec.replace(f"_{base_yr - pyr}", "")}'}, inplace=True)

    return fs_df_adj


def get_stocklist():
    df_krstock_kospi = fdr.StockListing('KOSPI')
    df_krstock_kosdaq = fdr.StockListing('KOSDAQ')
    df_krstock = pd.concat([df_krstock_kospi, df_krstock_kosdaq])
    df_krstock.reset_index(inplace=True, drop=True)
    df_krstock.rename(columns={'Code': 'Symbol'}, inplace=True)
    return df_krstock

def get_finance_info(match_krstock, yr, report_code):
    stock_finance = pd.DataFrame()
    except_list = {'Name': []}
    for fname in match_krstock['Name']:
        try:
            corp_finance = dart_fn.finstate_all(fname, yr, report_code)
            corp_finance['Name'] = fname
            stock_finance = pd.concat([stock_finance, corp_finance])
        except:
            except_list['Name'].append(fname)

    stock_finance.reset_index(inplace=True, drop=True)
    return stock_finance, except_list


def get_cap(base_day):
    date = str(base_day.year) + str(base_day.month).zfill(2) + str(base_day.day).zfill(2)
    limit_num = 1
    while limit_num < 10:
        market_info = stock.get_market_cap(date)
        if market_info['시가총액'].sum() != 0:
            break
        else:
            prev_date = base_day + relativedelta(days=-limit_num)
            date = str(prev_date.year) + str(prev_date.month).zfill(2) + str(prev_date.day).zfill(2)
            limit_num += 1

    market_info.reset_index(inplace=True)
    market_info.rename(columns={'단축코드': '종목코드', '상장주식수': 'Stocks'}, inplace=True)
    return market_info

In [47]:
base_day = datetime(2025, 6, 30)
market_info = get_cap(base_day)

api_key = "50424484a46daa88b34fcf875f40ca12b79e1fc1"
dart_fn = odr(api_key)

df_krstock = get_stocklist()
dart_list = dart_fn.corp_codes
dart_list.rename(columns={'stock_code': 'Symbol'}, inplace=True)
match_krstock = pd.merge(df_krstock, dart_list, on='Symbol', how='left')
match_krstock = match_krstock[match_krstock['corp_code'].isna() == False]

finance_df, except_list = get_finance_info(match_krstock[:5], '2023', '11013')

reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'


In [48]:
folder = r'C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT'
fs_df = make_finance_df(folder)

base_day = datetime(2023, 10, 26)

market_info = get_cap(base_day)

fs_df_adj = change_colname(fs_df, base_day.year - 1)

fs_df_adj_cap = pd.merge(fs_df_adj, market_info[['종목코드', '시가총액']], on='종목코드', how='left')

📂 전체 파일수: 620

🚀 2023 데이터 처리중...
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_01_재무상태표_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_01_재무상태표_연결_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_02_손익계산서_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_02_손익계산서_연결_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_03_포괄손익계산서_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_03_포괄손익계산서_연결_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_04_현금흐름표_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_04_현금흐름표_연결_20230503.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_05_자본변동표_20230503.txt


C:\Users\MetaM\AppData\Local\Temp\ipykernel_20352\3760070530.py:25: DtypeWarning: Columns (23,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(each_f, sep="\t", encoding='cp949')


📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2015_사업보고서_05_자본변동표_연결_20230503.txt


C:\Users\MetaM\AppData\Local\Temp\ipykernel_20352\3760070530.py:25: DtypeWarning: Columns (25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(each_f, sep="\t", encoding='cp949')


📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_01_재무상태표_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_01_재무상태표_연결_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_02_손익계산서_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_02_손익계산서_연결_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_03_포괄손익계산서_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_03_포괄손익계산서_연결_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_04_현금흐름표_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_04_현금흐름표_연결_20230119.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2016_1분기보고서_05_자본변동표_20230119.txt


...

C:\Users\MetaM\AppData\Local\Temp\ipykernel_20352\3760070530.py:25: DtypeWarning: Columns (21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(each_f, sep="\t", encoding='cp949')


📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_금융기타_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_금융기타_연결_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_보험_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_보험_연결_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_연결_20250606.txt


C:\Users\MetaM\AppData\Local\Temp\ipykernel_20352\3760070530.py:25: DtypeWarning: Columns (23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(each_f, sep="\t", encoding='cp949')


📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_은행_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_은행_연결_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_증권_20250606.txt
📂 파일 읽는중: C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_사업보고서_05_자본변동표_증권_연결_20250606.txt
🔍 con_df.shape=(5155048, 50), columns=['재무제표종류', '종목코드', '회사명', '시장구분', '업종', '업종명', '결산월', '결산기준일', '보고서종류', '통화', '항목코드', '항목명', '당기', '전기', '전전기', 'Unnamed: 15', 'Unnamed: 12', 'Unnamed: 14', 'Unnamed: 18', 'Unnamed: 13', 'Unnamed: 16', '당기 3분기말', '전기말', '전전기말', '당기 3분기 3개월', '당기 3분기 누적', '전기 3분기 3개월', '전기 3분기 누적', '당기 3분기', '전기 3분기', '당기 1분기말', '당기 1분기 3개월', '당기 1분기 누적', '전기 1분기 3개월', '전기 1분기 누적', '당기 1분기', '전기 1분기', '당기 반기말', '당기 반기 3개월', '당기 반기 누적', '전기 반기 3개월', '전기 반기 누적', '당기 반기', '전기 반기', '당기1분기', '전기1분기', '당기3분기', '전기3분기', '당기반기', '전기반기']
🔍 sol_df.shape=(5278270, 50), columns=[

MemoryError: Unable to allocate 304. MiB for an array with shape (2, 19899261) and data type object

In [41]:
file_path = r'C:\Users\MetaM\PycharmProjects\investment\DART_FS_TXT\2024_반기보고서_01_재무상태표_금융기타_20250605.txt'

df = pd.read_csv(file_path, sep="\t", encoding='cp949')

print(df.head())
print(df.columns)

            재무제표종류      종목코드      회사명        시장구분   업종     업종명  결산월  \
0   재무상태표, 기타 - 별도  [138930]  BNK금융지주  유가증권시장상장법인  649  기타 금융업   12   
1   재무상태표, 기타 - 별도  [138930]  BNK금융지주  유가증권시장상장법인  649  기타 금융업   12   
2   재무상태표, 기타 - 별도  [138930]  BNK금융지주  유가증권시장상장법인  649  기타 금융업   12   
3   재무상태표, 기타 - 별도  [138930]  BNK금융지주  유가증권시장상장법인  649  기타 금융업   12   
4   재무상태표, 기타 - 별도  [138930]  BNK금융지주  유가증권시장상장법인  649  기타 금융업   12   

        결산기준일  보고서종류   통화                                               항목코드  \
0  2024-06-30  반기보고서  KRW                           dart_CashAndDuefromBanks   
1  2024-06-30  반기보고서  KRW      dart_SecuritiesAtFairValueThroughProfitOrLoss   
2  2024-06-30  반기보고서  KRW  entity00858364_udf_BS_202311913543885OfAssetsA...   
3  2024-06-30  반기보고서  KRW  entity00858364_udf_BS_202391111717570OfAssetsA...   
4  2024-06-30  반기보고서  KRW                                   ifrs-full_Assets   

                          항목명             당기 반기말                전기말  전전기말  \
0              

In [36]:
filelist = glob(os.path.join(folder, "사업보고서*.txt"))
print("찾은 파일 목록:", filelist)

찾은 파일 목록: []
